# Foundations: The Agentic Systems Framework

**What you'll learn:**
- The spectrum of agentic complexity (single call → workflows → agents)
- The Augmented LLM — the universal building block
- When to use workflows vs. agents vs. a single LLM call
- The Agent-Computer Interface (ACI) — why tool design matters more than you think
- Three design principles for production agents

**Prerequisites:** Python 3.9+, `pip install anthropic`, `ANTHROPIC_API_KEY` environment variable set.

> *"The most successful implementations weren't using complex frameworks or specialized libraries — they were using simple, composable patterns."* — Anthropic, Building Effective Agents

## The Complexity Spectrum

Not every LLM application needs an agent. Anthropic categorizes agentic systems into a spectrum of increasing complexity:

| Level | Architecture | Control Flow | Best For |
|-------|-------------|-------------|----------|
| **Single LLM Call** | One prompt + retrieval/examples | None | Most tasks (start here!) |
| **Workflows** | Multiple LLM calls orchestrated by code | Developer-defined at design time | Well-defined, repeatable processes |
| **Agents** | LLM dynamically chooses actions + tools | Model-determined at runtime | Open-ended, unpredictable problems |

**The key distinction:** In workflows, YOU write the control flow. In agents, the MODEL decides what to do next.

### The Simplicity Imperative

The recommended progression:
1. Start with a single optimized LLM call (with retrieval + in-context examples)
2. Optimize with comprehensive evaluation
3. Add multi-step workflows only when simpler solutions fall short
4. Use autonomous agents only for truly open-ended problems

Each layer of complexity trades **latency**, **cost**, and **debuggability** for task performance. It must demonstrably earn its place.

## The Augmented LLM — The Building Block

Every node in every agentic system is fundamentally an **Augmented LLM**: a language model enhanced with three capabilities:

![The Augmented LLM — LLM with bidirectional connections to Retrieval, Tools, and Memory](assets/augmented_llm.webp)

| Augmentation | What it does | Example |
|-------------|-------------|--------|
| **Retrieval** | Model generates search queries to access external knowledge | RAG, vector DB lookups |
| **Tools** | Model selects and invokes structured API calls | Calculator, code execution, web search |
| **Memory** | Model decides what to retain across interactions | Conversation history, working state |

**Key principles:**
1. **Tailor capabilities** — Don't add all augmentations by default. Choose what fits the task.
2. **Provide clear interfaces** — Poor tool documentation degrades model performance.
3. **Composability** — Each LLM call in a workflow is itself an augmented LLM. The building block is recursive.

In [ ]:
import os
from anthropic import Anthropic

# Verify API connection
client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=100,
    messages=[{"role": "user", "content": "Say 'API connected!' in exactly two words."}]
)
print(f"✓ {response.content[0].text}")
print(f"  Model: {response.model}")
print(f"  Usage: {response.usage.input_tokens} input, {response.usage.output_tokens} output tokens")

### Demo: The Single Augmented LLM Call

Before building multi-step systems, let's see how powerful a **single** augmented LLM call can be. This uses Claude's native tool use — the model decides whether to call a tool based on the query.

In [ ]:
import json

# Define a simple tool — a calculator
tools = [
    {
        "name": "calculator",
        "description": "Performs arithmetic calculations. Use this for any math operation.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "The mathematical expression to evaluate, e.g. '(25 * 4) + 10'"
                }
            },
            "required": ["expression"]
        }
    }
]

def handle_tool_call(tool_name, tool_input):
    """Execute a tool call and return the result."""
    if tool_name == "calculator":
        try:
            result = eval(tool_input["expression"])
            return str(result)
        except Exception as e:
            return f"Error: {e}"
    return "Unknown tool"

# Ask a question that requires calculation
query = "If a company has 847 employees and wants to give each a $125 bonus, what's the total cost?"

print(f"Query: {query}\n")

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    tools=tools,
    messages=[{"role": "user", "content": query}]
)

# Process the response — model may use the tool or answer directly
for block in response.content:
    if block.type == "tool_use":
        print(f"→ Model chose tool: {block.name}")
        print(f"  Input: {block.input}")
        result = handle_tool_call(block.name, block.input)
        print(f"  Result: {result}")
        
        # Send tool result back for final answer
        final = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1024,
            tools=tools,
            messages=[
                {"role": "user", "content": query},
                {"role": "assistant", "content": response.content},
                {"role": "user", "content": [{"type": "tool_result", "tool_use_id": block.id, "content": result}]}
            ]
        )
        print(f"\n Final answer: {final.content[0].text}")
    elif block.type == "text":
        print(f"Direct answer: {block.text}")

## Decision Framework: Which Pattern Do You Need?

This is the most important section of this tutorial. Before building anything complex, use this framework:

### Step 1: Do you even need agentic systems?

Ask yourself:
- Can a single well-crafted prompt with good examples handle this? → **Stop here. Use a single call.**
- Does adding retrieval (RAG) to a single call solve it? → **Stop here. Use augmented single call.**

### Step 2: Workflows vs. Agents?

| Question | If YES → | If NO → |
|----------|----------|--------|
| Can you enumerate all subtasks in advance? | Workflow | Agent |
| Is the path through the task predictable? | Workflow | Agent |
| Do you need deterministic, repeatable behavior? | Workflow | Agent |
| Must the system adapt to unexpected situations? | Agent | Workflow |
| Is the number of steps unpredictable? | Agent | Workflow |

### Step 3: Which workflow pattern?

| Your situation | Pattern | Why |
|---------------|---------|-----|
| Task has fixed sequential steps | **Prompt Chaining** | Linear pipeline with quality gates |
| Inputs fall into distinct categories needing different handling | **Routing** | Classify once, optimize each path |
| Subtasks are independent, speed matters | **Parallelization** | Fan-out + aggregate |
| Can't predict subtasks until you see the input | **Orchestrator-Workers** | Dynamic decomposition |
| Output quality improves with iterative feedback | **Evaluator-Optimizer** | Generate → critique → refine loop |

In [ ]:
def recommend_pattern():
    """Interactive helper to recommend an agentic pattern based on your task."""
    
    print("=" * 60)
    print("  AGENTIC PATTERN RECOMMENDER")
    print("=" * 60)
    
    # Simulated decision logic (in practice, you'd use input())
    scenarios = [
        {
            "task": "Translate a document, then summarize it, then extract key terms",
            "single_call_works": False,
            "subtasks_known": True,
            "sequential": True,
            "recommendation": "Prompt Chaining",
            "reason": "Fixed sequential steps where each builds on the previous output"
        },
        {
            "task": "Handle customer support tickets (billing, technical, account issues)",
            "single_call_works": False,
            "subtasks_known": True,
            "sequential": False,
            "recommendation": "Routing",
            "reason": "Distinct input categories that benefit from specialized handling"
        },
        {
            "task": "Review code for security, performance, and style simultaneously",
            "single_call_works": False,
            "subtasks_known": True,
            "sequential": False,
            "recommendation": "Parallelization (Sectioning)",
            "reason": "Independent subtasks that benefit from focused attention"
        },
        {
            "task": "Write marketing copy for a product (unknown angles until you see it)",
            "single_call_works": False,
            "subtasks_known": False,
            "sequential": False,
            "recommendation": "Orchestrator-Workers",
            "reason": "Subtasks determined at runtime based on the specific input"
        },
        {
            "task": "Generate production-quality code that passes all tests",
            "single_call_works": False,
            "subtasks_known": True,
            "sequential": True,
            "recommendation": "Evaluator-Optimizer",
            "reason": "Clear evaluation criteria + iterative refinement adds demonstrable value"
        },
        {
            "task": "Answer a factual question with context",
            "single_call_works": True,
            "subtasks_known": True,
            "sequential": False,
            "recommendation": "Single LLM Call (no agentic system needed)",
            "reason": "A well-crafted prompt with retrieval handles this perfectly"
        },
    ]
    
    for scenario in scenarios:
        print(f"\n{'─' * 60}")
        print(f"  Task: {scenario['task']}")
        print(f"  → Recommendation: {scenario['recommendation']}")
        print(f"  → Why: {scenario['reason']}")
    
    print(f"\n{'═' * 60}")
    print("  Remember: Start simple. Earn complexity.")
    print("═" * 60)

recommend_pattern()

## Agent-Computer Interface (ACI) — A Preview

The **Agent-Computer Interface** is the agent-facing analog of Human-Computer Interaction (HCI). Just as HCI invests in making interfaces intuitive for humans, ACI invests in making tools intuitive for LLMs.

**Core insight from Anthropic:** In building SWE-bench agents, more time was spent optimizing tool interfaces than the overall prompt. Poor ACI is the #1 root cause of agent failures (often misattributed to "model limitations").

### The 5 ACI Principles

| # | Principle | What it means |
|---|-----------|---------------|
| 1 | **Empathize with the model** | If a tool's usage isn't obvious from its description, it won't be obvious to the model either |
| 2 | **Format for LLM strengths** | Use formats close to training data (markdown > JSON for code, XML for structured output) |
| 3 | **Give thinking space** | Ensure enough tokens for reasoning before committing to constrained outputs |
| 4 | **Poka-yoke (error-proofing)** | Design arguments so mistakes are structurally difficult (e.g., absolute paths only) |
| 5 | **Document like onboarding** | Include examples, edge cases, boundaries — as if training a junior developer |

We'll see ACI in action throughout the pattern notebooks, especially in [04_orchestrator_workers.ipynb](04_orchestrator_workers.ipynb) and [06_autonomous_agents.ipynb](06_autonomous_agents.ipynb).

## Three Design Principles for Production Agents

These principles from Anthropic's production experience guide every architectural decision:

### 1. Simplicity
> The most successful production implementations use simple, composable patterns rather than complex frameworks.

- Prefer direct API usage over heavy frameworks
- If using a framework, understand what's under the hood
- Don't add orchestration unless single-call approaches have been measured and found wanting

### 2. Transparency
> Explicitly show the agent's planning steps to enable human oversight.

- Agents can pause for human feedback at checkpoints
- Autonomous operation creates compounding error risk
- Making reasoning visible enables debugging, trust-building, and early termination

### 3. Careful ACI Design
> Tool design deserves equal investment to prompt engineering.

- Test tool usage with many example inputs
- Iterate on descriptions and parameter design
- Error-proof tool arguments (poka-yoke)
- Choose formats that play to LLM strengths

---

## What's Next

Now that you understand the conceptual framework, it's time to implement patterns:

| Next notebook | What you'll build |
|--------------|-------------------|
| [01_prompt_chaining.ipynb](01_prompt_chaining.ipynb) | Sequential steps with quality gates |
| [02_routing.ipynb](02_routing.ipynb) | Classify and dispatch to specialists |
| [03_parallelization.ipynb](03_parallelization.ipynb) | Fan-out for speed and confidence |
| [04_orchestrator_workers.ipynb](04_orchestrator_workers.ipynb) | Dynamic task decomposition |
| [05_evaluator_optimizer.ipynb](05_evaluator_optimizer.ipynb) | Iterative refinement loops |
| [06_autonomous_agents.ipynb](06_autonomous_agents.ipynb) | Full agent loop with all principles applied |